# Quera - Intent Classifier Training

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline
import joblib

## 1. Load and Randomize Data

In [ ]:
# Load and randomize data
df = pd.read_csv("training_data.csv")
print(f"Total samples: {len(df)}")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.head()

## 2. Train-Test Split (80/20)

In [ ]:
# Stratified 80-20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

## 3. Define and Train Models

In [ ]:
# Define models
pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

pipeline_svc = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ('clf', CalibratedClassifierCV(LinearSVC(random_state=42, dual='auto'), method='sigmoid', cv=5))
])

In [ ]:
# Train and Evaluate Logistic Regression
pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)
print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

In [ ]:
# Train and Evaluate LinearSVC
pipeline_svc.fit(X_train, y_train)
y_pred_svc = pipeline_svc.predict(X_test)
print("=== LinearSVC (Calibrated) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_svc))
print(classification_report(y_test, y_pred_svc))

## 4. Save Model

In [ ]:
# Save best model
# Assuming LinearSVC was better based on your runs
joblib.dump(pipeline_svc, "classifier.joblib")
print("Saved winning model to classifier.joblib")